In [1]:
from pyspark.sql import SparkSession

# 1. Forzamos el cierre de sesiones previas para evitar conflictos de configuración
try:
    spark.stop()
except:
    pass

# 2. Creamos la nueva sesión incluyendo el conector de MongoDB
# Usamos el paquete org.mongodb.spark:mongo-spark-connector para que Spark sepa leer la DB
spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/Mercado_Laboral_TI.AmazonLaptops") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

# 3. Intentamos cargar los datos
try:
    df = spark.read.format("mongodb").load()
    
    # Verificamos si se cargó algo
    conteo = df.count()
    print(f"Éxito total: {conteo} registros cargados.")
    
    if conteo > 0:
        df.show(5)
    else:
        print("La colección está vacía. Verifica que el scraper haya guardado los datos.")

except Exception as e:
    print("Error al cargar los datos de MongoDB:")
    print(e)

Éxito total: 131 registros cargados.
+--------------------+-------------------+------------------+--------------------+-----+
|                 _id|      fecha_captura|             grupo|       identificador|valor|
+--------------------+-------------------+------------------+--------------------+-----+
|69e28f7996bce49be...|2026-04-17 19:51:31|Mercado Laboral TI|15.6" FHD Laptop ...|399.0|
|69e28f7996bce49be...|2026-04-17 19:51:32|Mercado Laboral TI|Laptop 15.6 Inch ...|349.0|
|69e28f7996bce49be...|2026-04-17 19:51:32|Mercado Laboral TI|Laptop Ultralight...|369.0|
|69e28f7996bce49be...|2026-04-17 19:51:32|Mercado Laboral TI|15.6 Inch Laptop,...|289.0|
|69e28f7996bce49be...|2026-04-17 19:51:32|Mercado Laboral TI|Lenovo IdeaPad Sl...|449.0|
+--------------------+-------------------+------------------+--------------------+-----+
only showing top 5 rows



In [2]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+-----------------+
|    marca|  precio_promedio|
+---------+-----------------+
|      MSI|           1558.0|
|       LG|           1305.0|
|     Acer|           848.25|
|   Lenovo| 833.304347826087|
|     ASUS|            827.0|
|     DELL|            779.5|
|Microsoft|            770.0|
| ACEMAGIC|            514.0|
|  Lenovo,|            492.5|
|     acer|           461.75|
|    Ryzen|            439.0|
|       16|            439.0|
|        2|            419.0|
|   Laptop|397.0769230769231|
|      GMR|            393.0|
|    15.6"|            374.0|
|     2021|            353.0|
|       HP|            351.0|
|     15.6|           340.25|
|  domyfan|            309.0|
+---------+-----------------+
only showing top 20 rows

